# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv') # tu código aquí
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv') # tu código aquí
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv') # tu código aquí

In [3]:
#Explorar datasets
orders.info()
catalog.info()
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
--- 

In [4]:
#Información de las tablas
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [5]:
catalog.head(10)

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


In [6]:
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [7]:
#Revisar dubplicados
orders.duplicated().sum()

100

In [8]:
catalog.duplicated().sum()

0

In [9]:
marketing.duplicated().sum()

0

In [10]:
#Resumen estadístico
orders.describe().round()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.0,25050.0,25050.0,25100.0
mean,7.0,259.0,5.0,2073.0
std,296.0,139.0,5.0,98950.0
min,-2.0,20.0,0.0,-493.0
25%,1.0,138.0,0.0,181.0
50%,2.0,259.0,0.0,342.0
75%,2.0,380.0,10.0,519.0
max,20000.0,500.0,15.0,8840200.0


In [11]:
orders.select_dtypes(include="object").describe()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto
count,25100,25100,25100,24800,25080,25070,25070,25020
unique,25000,7642,181,6,2,3,7,3
top,order_14562,user_7769,2025-06-25,Colombia,desktop,social,Vacuum-Pro-Black,Hogar
freq,2,11,176,7520,12759,8428,4199,8385


In [12]:
#Revisar cantidad de valores nulo o faltantes
orders.isna().sum().sort_values(ascending=False)

pais                  300
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
fuente_referencia      30
nombre_producto        30
dispositivo            20
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
monto_total             0
dtype: int64

In [13]:
#Analizar variables categoricas 
orders["pais"].value_counts(dropna=False)

Colombia     7520
Mexico       7502
Argentina    7291
mexico        865
colombia      823
argentina     799
NaN           300
Name: pais, dtype: int64

In [14]:
#Analizar outliers 

orders_outliers= orders[orders["cantidad"]>=5]
print(f"Registros con cantidad vendida mayor a 5: {orders_outliers.shape}")
orders_outliers

Registros con cantidad vendida mayor a 5: (10, 12)


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
3521,order_3521,user_5812,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,43.14,0.0,431400.0
3522,order_3522,user_3575,2025-03-29,Argentina,desktop,social,Laptop-Gaming-16GB,Electronica,10000.0,280.55,0.0,2805500.0
3586,order_3586,user_3380,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,490.35,0.0,4903500.0
3643,order_3643,user_4440,2025-01-07,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,10000.0,238.15,0.0,2381500.0
3656,order_3656,user_884,2025-01-01,Argentina,mobile,organic,Laptop-Gaming-16GB,Electronica,20000.0,297.66,0.0,5953200.0
3668,order_3668,user_7270,2025-06-24,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,20000.0,348.31,0.0,6966200.0
3689,order_3689,user_6566,2025-06-16,mexico,desktop,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,87.69,0.0,876900.0
3722,order_3722,user_4723,2025-05-09,Argentina,mobile,paid_search,Laptop-Gaming-16GB,Electronica,20000.0,442.01,0.0,8840200.0
3726,order_3726,user_2536,2025-02-12,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,20000.0,290.85,0.0,5817000.0
3748,order_3748,user_7096,2025-02-23,Mexico,desktop,organic,Laptop-Gaming-16GB,Electronica,10000.0,336.93,0.0,3369300.0


In [15]:
outlier_index= orders_outliers.index

print(outlier_index)

Int64Index([3521, 3522, 3586, 3643, 3656, 3668, 3689, 3722, 3726, 3748], dtype='int64')


In [16]:
#1)Existen valores faltantes tanto en columnas catégoricas como en las númericas, en el dataset de Organic(610 missing values) y marketing (101 missing values).
#2)Existen 100 valores duplicados en la tabla de organic que se deben eliminar.
#3)Existen valores negativos en columnas númericas(no deben existir).
#4)Existen outliers en la tabla de orders. 
#5)Existen valores inconsistentes en columnas categoricas que hay que corregir.

<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class="tocSkip"></a><br>
<b>Éxito</b> - Realizaste una exploración inicial completa de los datasets, identificando valores faltantes, duplicados, inconsistencias y posibles anomalías que serán fundamentales para garantizar la calidad de los datos antes de avanzar con el análisis de negocio.
</div>

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [17]:
# tu código aquí
orders_clean = orders.copy()
catalog_clean = catalog.copy()
marketing_clean = marketing.copy()

In [18]:
#Convertir fechas al formato correcto 
orders_clean["fecha_hora_pedido"]= pd.to_datetime(orders_clean["fecha_hora_pedido"])
marketing_clean["fecha"]= pd.to_datetime(marketing_clean["fecha"])

In [19]:
#Varialbes númericas-Convertir negativos a positivos
orders_clean["cantidad"] = orders_clean["cantidad"].abs()
orders_clean["monto_total"] = orders_clean["monto_total"].abs()

In [20]:
orders_clean.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.093134,259.305549,4.500798,2.072771e+03
std,296.276993,138.726461,5.223010,9.894995e+04
min,1.000000,20.030000,0.000000,5.240000e+00
25%,1.000000,138.377500,0.000000,1.805925e+02
50%,2.000000,258.715000,0.000000,3.418050e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [21]:
#Eliminar duplicados 
orders_clean = orders_clean.drop_duplicates()
print("Duplicados después de limpiar:", orders_clean.duplicated().sum())

Duplicados después de limpiar: 0


In [22]:
#Rellenar país usando compras anteriores o posteriores del mismo usuario.
orders["pais"] = orders.groupby("id_usuario")["pais"].transform(lambda x: x.ffill().bfill())
print(orders['pais'].isnull().sum())

19


In [23]:
#Rellenar valores faltantes de canal con la información de id_campaña 
marketing_clean["canal"]= marketing_clean["canal"].fillna(marketing_clean["id_campaña"].str.split("_").str[0])

print(marketing_clean["canal"].isnull().sum())

0


In [24]:
marketing_clean["canal"]= marketing_clean["canal"].replace("paid", "paid_search")

print(marketing_clean["canal"].value_counts())

social         540
organic        540
paid_search    540
Name: canal, dtype: int64


In [25]:
#Eliminar valunes nulos 
orders_clean = orders.dropna(subset=['pais', 'precio_unitario', 'cantidad', 'monto_descuento','fuente_referencia'])
print(f"Filas finales: {len(orders_clean)}")

Filas finales: 25001


In [26]:
#Eliminar outliers
orders_clean = orders_clean.drop(index=outlier_index)
print(f"Filas finales: {len(orders_clean)}")

Filas finales: 24991


In [27]:
orders_clean["pais"]= orders_clean["pais"].str.strip().str.title()

In [28]:
orders_clean["pais"].value_counts()

Mexico       8433
Colombia     8401
Argentina    8157
Name: pais, dtype: int64

In [29]:
orders_clean["nombre_producto"].value_counts()

Vacuum-Pro-Black        4189
Blender-XL-Red          4188
Jacket-Winter-M         4180
Sneakers-Urban-42       4140
Laptop-Gaming-16GB      2777
Tablet-Standard-64GB    2773
Phone-Pro-128GB         2744
Name: nombre_producto, dtype: int64

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [30]:
# exportar datasets
orders_clean.to_csv('orders_clean.csv', index=False)
catalog_clean.to_csv('catalog_clean.csv', index=False)
marketing_clean.to_csv('marketing_clean.csv', index=False)

In [31]:
pd.read_csv('orders_clean.csv').shape

(24991, 12)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [32]:
# tu código aquí
# Unir catalog con orders_clean
orders_cost = orders_clean.merge(catalog[['nombre_producto', 'costo_unitario']], 
                                       on='nombre_producto', how='left')

Ingreso_Total= orders_clean["monto_total"].sum()
Costo_Total= (orders_cost['costo_unitario'] * orders_cost['cantidad']).sum()
Gasto_marketing= marketing_clean["gasto"].sum()
profit = Ingreso_Total - (Costo_Total + Gasto_marketing)

print(f"Ingreso total:       ${Ingreso_Total:,.2f}")
print(f"Costo total:         ${Costo_Total:,.2f}")
print(f"Gasto marketing:     ${Gasto_marketing:,.2f}")
print(f"Profit:              ${profit:,.2f}")


Ingreso total:       $9,638,853.49
Costo total:         $3,842,567.81
Gasto marketing:     $2,871,843.53
Profit:              $2,924,442.15


In [33]:
Ticket_promedio =  orders_clean['monto_total'].mean()
Cantidad_promedio = orders_clean["cantidad"].mean()
Producto_mas_vendidos= orders_clean.groupby("nombre_producto")["cantidad"].sum().sort_values(ascending=False)
Gasto_mkt_canal= marketing_clean.groupby("canal")["gasto"].sum().sort_values(ascending=False)

print(f"Cantidad promedio por orden: {Cantidad_promedio:,.2f}")
print("\nProductos más vendidos:",Producto_mas_vendidos )
print("\nGasto en marketing por canal:", Gasto_mkt_canal)


Cantidad promedio por orden: 1.50

Productos más vendidos: nombre_producto
Vacuum-Pro-Black        6312.0
Blender-XL-Red          6298.0
Jacket-Winter-M         6278.0
Sneakers-Urban-42       6189.0
Laptop-Gaming-16GB      4216.0
Tablet-Standard-64GB    4166.0
Phone-Pro-128GB         4144.0
Name: cantidad, dtype: float64

Gasto en marketing por canal: canal
social         976818.37
organic        972650.96
paid_search    922374.20
Name: gasto, dtype: float64


In [34]:
#Los clientes típicamente compran 1 o 2 productos por orden.
#Productos más vendidos: El top 3 son del hogar/moda (Vacuum, Blender, Jacket) con ~6,300 unidades cada uno. Electrónica (Laptop, Tablet, Phone) vende considerablemente menos ~4,100 unidades — pero probablemente compensan con precio unitario más alto

#Gasto en marketing por canal:Los 3 canales están muy balanceados (~$970k cada uno)social lidera por poco, pero la diferencia es mínima

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [35]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [36]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [37]:
query_events = '''
SELECT DISTINCT nombre_evento FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head(10)

,nombre_evento
0,add_payment_info
1,first_visit
2,begin_checkout
3,add_to_cart
4,select_item
5,purchase


In [38]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT 
      nombre_evento,
      COUNT (DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY 1
ORDER BY 2 DESC
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [39]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
SELECT
    'first_visit → add_to_cart'     AS paso,
    100 * COUNT(DISTINCT CASE WHEN nombre_evento = 'add_to_cart'      THEN id_usuario END)
        / COUNT(DISTINCT CASE WHEN nombre_evento = 'first_visit'      THEN id_usuario END) AS tasa_conversion

FROM events

UNION ALL SELECT
    'add_to_cart → select_item',
    100 * COUNT(DISTINCT CASE WHEN nombre_evento = 'select_item'      THEN id_usuario END)
        / COUNT(DISTINCT CASE WHEN nombre_evento = 'add_to_cart'      THEN id_usuario END)
FROM events

UNION ALL SELECT
    'select_item → begin_checkout',
    100 * COUNT(DISTINCT CASE WHEN nombre_evento = 'begin_checkout'   THEN id_usuario END)
        / COUNT(DISTINCT CASE WHEN nombre_evento = 'select_item'      THEN id_usuario END)
FROM events

UNION ALL SELECT
    'begin_checkout → add_payment_info',
    100 * COUNT(DISTINCT CASE WHEN nombre_evento = 'add_payment_info' THEN id_usuario END)
        / COUNT(DISTINCT CASE WHEN nombre_evento = 'begin_checkout'   THEN id_usuario END)
FROM events

UNION ALL SELECT
    'add_payment_info → purchase',
    100 * COUNT(DISTINCT CASE WHEN nombre_evento = 'purchase'         THEN id_usuario END)
        / COUNT(DISTINCT CASE WHEN nombre_evento = 'add_payment_info' THEN id_usuario END)
FROM events
ORDER BY tasa_conversion ASC;


'''



conversion = pd.read_sql(query_conversion, con=engine)
conversion

,paso,tasa_conversion
0,begin_checkout → add_payment_info,86
1,select_item → begin_checkout,95
2,first_visit → add_to_cart,97
3,add_to_cart → select_item,99
4,add_payment_info → purchase,99


In [40]:
#begin_checkout → add_payment_info con solo 86% de conversión  

In [41]:
#Tasa de conversión final 
query_conversion = '''
SELECT 
      COUNT (DISTINCT (CASE WHEN nombre_evento= 'first_visit' THEN id_usuario END))AS primera_visita_usuario,
100*COUNT(DISTINCT (CASE WHEN nombre_evento = 'add_to_cart' THEN id_usuario END))/COUNT (DISTINCT (CASE WHEN nombre_evento= 'first_visit' THEN id_usuario END)) AS retencion_add_to_cart,
100*COUNT(DISTINCT (CASE WHEN nombre_evento = 'select_item' THEN id_usuario END ))/COUNT (DISTINCT (CASE WHEN nombre_evento= 'first_visit' THEN id_usuario END)) AS retencion_select_item,
100*COUNT(DISTINCT (CASE WHEN nombre_evento = 'begin_checkout' THEN id_usuario END ))/COUNT (DISTINCT (CASE WHEN nombre_evento= 'first_visit' THEN id_usuario END)) AS retencion_checkout,
100*COUNT(DISTINCT (CASE WHEN nombre_evento = 'add_payment_info' THEN id_usuario END ))/COUNT (DISTINCT (CASE WHEN nombre_evento= 'first_visit' THEN id_usuario END)) AS retencion_payment_info,
100*COUNT(DISTINCT (CASE WHEN nombre_evento = 'purchase' THEN id_usuario END ))/COUNT (DISTINCT (CASE WHEN nombre_evento= 'first_visit' THEN id_usuario END)) AS retencion_purchase

FROM events
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,primera_visita_usuario,retencion_add_to_cart,retencion_select_item,retencion_checkout,retencion_payment_info,retencion_purchase
0,7796,97,97,92,80,80


In [42]:
#Tasa de conversión final = 80% 

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [43]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [44]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT * 
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [45]:
# Retención por cohortes
# ======================
from sqlalchemy import text
query_cohort_retention_final = '''
WITH cohorte_base AS (
    SELECT
        u.id_usuario,
        DATE_TRUNC('month', CAST(u.fecha_registro AS DATE)) AS cohorte,
        a.dias_despues_registro,
        a.activo
    FROM users u
    LEFT JOIN user_activity a ON u.id_usuario = a.id_usuario
),
cohorte_size AS (
    SELECT
        cohorte,
        COUNT(DISTINCT id_usuario) AS clientes_iniciales
    FROM cohorte_base
    GROUP BY cohorte
),
retencion AS (
    SELECT
        cohorte,
        COUNT(DISTINCT CASE 
            WHEN dias_despues_registro BETWEEN 1 AND 7 
            AND activo = 1 THEN id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE 
            WHEN dias_despues_registro BETWEEN 8 AND 14 
            AND activo = 1 THEN id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE 
            WHEN dias_despues_registro BETWEEN 15 AND 21 
            AND activo = 1 THEN id_usuario END) AS retenido_w3
    FROM cohorte_base
    GROUP BY cohorte
)
SELECT
    TO_CHAR(r.cohorte, 'YYYY-MM') AS cohorte,
    c.clientes_iniciales,
    ROUND(100.0 * r.retenido_w1 / c.clientes_iniciales, 1) AS semana_1,
    ROUND(100.0 * r.retenido_w2 / c.clientes_iniciales, 1) AS semana_2,
    ROUND(100.0 * r.retenido_w3 / c.clientes_iniciales, 1) AS semana_3
FROM retencion r
JOIN cohorte_size c ON r.cohorte = c.cohorte
ORDER BY r.cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,clientes_iniciales,semana_1,semana_2,semana_3
0,2025-01,1627,42.8,41.1,40.3
1,2025-02,1444,42.3,42.2,44.0
2,2025-03,1636,41.4,43.1,42.2
3,2025-04,1606,42.3,43.4,41.3
4,2025-05,1687,41.2,40.1,41.8


In [46]:
#1. La retención es muy estable entre semanas — los usuarios que regresan en semana 1 tienden a seguir activos en semana 2 y 3. No hay caída drástica.
#2. Solo ~42% de usuarios regresa — más de la mitad de los usuarios no vuelve después de registrarse. Hay oportunidad de mejora con estrategias de reactivación (emails, notificaciones, ofertas).
#3. No hay diferencia significativa entre cohortes — el producto se comporta igual mes a mes, lo que indica consistencia pero también que no ha habido mejoras que aumenten la retención.

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** No hay diferencia en la tasa de conversión entre control y tratamiento.

     
   - **H₁ (Hipótesis alternativa):** Sí hay diferencia en la tasa de conversión. 
   
**Test estadístico:** Chi-cuadrado. Compararemos 2 variales categóricas (grupo y si compró o no) 

**Nivel de significancia alpha:** 0.05 -> Estándar en la indrustrial (aceptamos 5% de probabilidad de error)

In [47]:
# tu código aquí
df_experiment= pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
df_experiment.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [48]:
df_experiment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


In [49]:
df_experiment['variante'].value_counts()

tratamiento    5035
control        4965
Name: variante, dtype: int64

In [50]:
df_experiment['convirtio'].value_counts()

0    8401
1    1599
Name: convirtio, dtype: int64

In [51]:
from scipy import stats

# Separar grupos
control = df_experiment[df_experiment['variante'] == 'control']['convirtio']
tratamiento = df_experiment[df_experiment['variante'] == 'tratamiento']['convirtio']

# Tasas de conversión
tasa_control     = control.mean()
tasa_tratamiento = tratamiento.mean()

print(f"Tasa conversión CONTROL:     {tasa_control:.4f} ({tasa_control*100:.2f}%)")
print(f"Tasa conversión TRATAMIENTO: {tasa_tratamiento:.4f} ({tasa_tratamiento*100:.2f}%)")
print(f"Diferencia:                  {(tasa_tratamiento - tasa_control)*100:.2f} puntos porcentuales")

Tasa conversión CONTROL:     0.1569 (15.69%)
Tasa conversión TRATAMIENTO: 0.1629 (16.29%)
Diferencia:                  0.60 puntos porcentuales


In [52]:
#Test Chi-cuadrado
tabla_contingencia = pd.crosstab(df_experiment['variante'], df_experiment['convirtio'])
print(tabla_contingencia)

chi2, p_valor, dof, expected = stats.chi2_contingency(tabla_contingencia)

print(f"\nChi2:    {chi2:.4f}")
print(f"P-valor: {p_valor:.4f}")

convirtio       0    1
variante              
control      4186  779
tratamiento  4215  820

Chi2:    0.6178
P-valor: 0.4319


In [53]:
#Interpretación
alpha = 0.05

if p_valor < alpha:
    print(f" p-valor ({p_valor:.4f}) < alpha ({alpha})")
    print("→ RECHAZAMOS H₀: El cambio en la UI SÍ tiene impacto significativo")
else:
    print(f" p-valor ({p_valor:.4f}) >= alpha ({alpha})")
    print("→ NO rechazamos H₀: El cambio en la UI NO tiene impacto significativo")

 p-valor (0.4319) >= alpha (0.05)
→ NO rechazamos H₀: El cambio en la UI NO tiene impacto significativo


In [54]:
#NO rechazamos H₀
#Aunque el grupo de tratamiento muestra una tasa de conversión ligeramente mayor (16.29% vs 15.69%), la diferencia de 0.60 puntos porcentuales NO es estadísticamente significativa.
#Con un p-valor de 0.4319, hay un 43% de probabilidad de que esta diferencia sea simplemente por azar, muy por encima del 5% permitido.

<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class="tocSkip"></a><br>
<b>Éxito</b> - El planteamiento de hipótesis, la selección del test Chi-cuadrado, el cálculo de las tasas de conversión y la interpretación del p-valor están correctamente desarrollados, permitiendo concluir de forma fundamentada que el cambio en la interfaz no produjo un impacto estadísticamente significativo en la conversión.
</div>

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [55]:
# (https://drive.google.com/file/d/1m7HQ2N75iuVw52xH-NcgIM9xZoqEbzqB/view?usp=sharing)
# link de power bi o tableau
# link de one drive / google drive

✏️ Comparto el enlace a mi archivo .pbix en Google Drive:
    
https://drive.google.com/file/d/1m7HQ2N75iuVw52xH-NcgIM9xZoqEbzqB/view?usp=sharing